# 05. 인사이트 및 실행 방안 (Insights & Actions)

## 목표
- 전체 분석 결과 종합
- 핵심 인사이트 도출
- 실행 가능한 권고사항 제시

## ULTRA-THINK Framework: A - Anticipate (예측하기) & T - Tell Story (스토리텔링)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 로딩

In [ ]:
# 전처리된 데이터
df = pd.read_csv("../data/processed/merged_hourly_202509.csv", encoding="utf-8")
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])

# 우천 감소율 데이터
rain_impact = pd.read_csv("../data/processed/rain_impact_by_station.csv", encoding="utf-8", index_col=0)
rain_impact = rain_impact.squeeze()

print("데이터 로딩 완료")

## 2. 핵심 인사이트 요약

### 인사이트 1: 강수 임계치
**일강수량 10mm+에서 이용량이 비선형적으로 급감**

In [ ]:
# 강수 구간별 평균 이용량
rain_bins_avg = df.groupby('강수_구간', observed=True)['전체_건수'].mean()
print("\n=== 강수 구간별 평균 이용량 ===")
print(rain_bins_avg)

# 감소율 계산
baseline = rain_bins_avg.iloc[0]  # 0mm(맑음)
for idx, val in rain_bins_avg.items():
    decrease = ((val - baseline) / baseline) * 100
    print(f"{idx}: {decrease:+.1f}%")

### 인사이트 2: 시간대 민감도
**퇴근 시간대와 주말 낮에 우천 감소폭이 가장 큼**

In [ ]:
# 시간대 구분별 맑은 날 vs 비 오는 날
time_period_comparison = df.groupby(['시간대_구분', '비여부'])['전체_건수'].mean().unstack()
time_period_comparison['감소율(%)'] = ((time_period_comparison[1] - time_period_comparison[0]) / time_period_comparison[0] * 100)

print("\n=== 시간대별 우천 감소율 ===")
print(time_period_comparison.sort_values('감소율(%)'))

### 인사이트 3: 공간 유형 차이
**환승형 vs 공원/여가형 대여소의 우천 민감도 차이**

In [ ]:
# TOP 10 / LOW 10 대여소
print("\n=== 우천 감소율 TOP 10 (가장 민감) ===")
print(rain_impact.head(10))

print("\n=== 우천 감소율 LOW 10 (가장 둔감) ===")
print(rain_impact.tail(10))

### 인사이트 4: 복합기상 영향
**비(≥10mm) + 풍속(≥5m/s) 동시 발생 시 최저 이용**

In [ ]:
# 복합 조건 분석
df['풍속_구간'] = pd.cut(df['평균풍속(m/s)'], bins=[0, 3, 5, 10], labels=['약함(<3)', '보통(3-5)', '강함(5+)'])

complex_condition = df.groupby(['강수_구간', '풍속_구간'], observed=True)['전체_건수'].mean().unstack()

print("\n=== 강수 × 풍속 조합별 평균 이용량 ===")
print(complex_condition)

# 히트맵
plt.figure(figsize=(10, 6))
sns.heatmap(complex_condition, annot=True, fmt='.1f', cmap='YlOrRd_r', cbar_kws={'label': '평균 이용 건수'})
plt.title('강수량 × 풍속 조합별 이용량', fontsize=14, fontweight='bold')
plt.xlabel('풍속 구간', fontsize=12)
plt.ylabel('강수 구간', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/heatmap_rain_wind.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. 실행 가능한 권고사항

### 단기 (1~3개월)

In [ ]:
short_term = """
### 단기 실행 방안 (1~3개월)

1. 기상 연동형 알림 시스템 구축
   - 조건: 일강수량 ≥ 10mm 또는 평균풍속 ≥ 5m/s
   - 실행: 앱 푸시 알림 + 대여소 안내판 자동 메시지
   - 내용: "우천으로 인한 이용 주의" + 안전 운행 가이드

2. 우선순위 운영 대상 선정
   - 대상: 우천 감소율 TOP 20 대여소
   - 실행: 비 예보 시 회수/정비 일정 사전 조정
   - 효과: 불필요한 재배치 작업 10~15% 절감 예상

3. 전광판·배너 메시지 운영
   - 내용: "우천 시 제동력 저하, 미끄럼 주의"
   - 부가: 비닐 커버 비치 (주요 대여소 시범 운영)
"""

print(short_term)

### 중기 (3~6개월)

In [ ]:
mid_term = """
### 중기 실행 방안 (3~6개월)

1. 경량 차양막 시범 설치
   - 대상: 우천 민감 TOP 10 대여소
   - 기대 효과: 소나기 시 이용 지속성 향상
   - C/B 분석 후 단계적 확대

2. 강수 예보 기반 경로 최적화
   - 회수 차량 이동 경로를 기상 예보에 따라 동적 조정
   - 예상 효과: 이동거리 10~15% 절감

3. 환승형 대여소 중심 재배치
   - 우천 시에도 이용률이 높은 지하철 인접 대여소 중심
   - 공원/여가형 대여소는 회수 우선순위 상향
"""

print(mid_term)

### 장기 (6개월 이상)

In [ ]:
long_term = """
### 장기 실행 방안 (6개월 이상)

1. 운영 규칙화
   - 월별 강수/풍속 임계치 기반 운영지침 표준화
   - 계절별 날씨 패턴 반영한 운영 매뉴얼 수립

2. 시설 투자 로드맵
   - 차양막·배수·노면 개선의 C/B 분석
   - 우선순위 기반 단계적 확대

3. 위험 날씨 대응 시뮬레이션
   - 극심한 기상 조건 시 일부 대여소 임시 폐쇄 검토
   - 안전성 향상 및 민원 감소 지표 모니터링

4. 우천 주간 정비·점검 효율화
   - 집중호우 예상 주간을 '정비 집중 주간'으로 운영
   - 배터리 교체, 부품 수리 등 집중 투입
   - 운영 공백 최소화 및 생산성 향상
"""

print(long_term)

## 4. 최종 스토리텔링

In [ ]:
story = """
=================================================================
비와 바람, 따릉이의 가장 큰 적!
날씨에 흔들리는 도시의 자전거
=================================================================

## 도입
2025년 9월, 서울은 이례적인 집중호우를 겪었습니다.
출퇴근길 시민들의 발이 되어온 따릉이는 이 기간 동안 어떤 변화를 겪었을까요?

## 주요 발견

**발견 1: 강수 임계치**
일강수량 10mm 이상일 때, 전체 이용량이 평균 대비 급격히 감소합니다.
특히 30mm 이상의 폭우 시에는 이용량이 거의 절반 수준으로 떨어집니다.

**발견 2: 시간대별 차이**
출근(7-9시) 시간대는 비가 와도 상대적으로 이용이 유지되지만,
주말 낮(12-17시)에는 급격히 감소합니다.
퇴근 시간대(18-20시)도 감소폭이 큽니다.

**발견 3: 공간 유형의 차이**
공원형 대여소(여의도, 뚝섬)는 비 오는 날 이용이 거의 '제로'에 가깝지만,
지하철 환승형 대여소(신림, 홍대입구)는 유지율이 높습니다.

**발견 4: 복합기상 효과**
비(≥10mm) + 풍속(≥5m/s) 동시 발생 시 이용량이 최저 수준으로 떨어집니다.
'비 + 바람' 복합조건이 최악의 조합입니다.

## 결론
날씨는 단순한 불편 요소가 아니라, 이용 패턴을 결정짓는 핵심 변수입니다.
특히 비와 바람이 겹치면 이용량이 급락하는 것을 확인했습니다.
집중호우 주간은 '정비·점검 효율화 주간'으로 전환하면 운영 생산성을 높일 수 있습니다.

## 권고사항
1. **기상 정보 연동형 알림 시스템 구축**
   비 예보 시, 앱/안내판을 통해 실시간 이용 주의 안내 제공

2. **날씨 민감형 대여소 관리 우선순위 설정**
   우천 시 급감 대여소를 중심으로 회수·정비 일정을 사전 조정

3. **차양막 설비 시범 설치**
   상위 10개 우천 취약 대여소를 선정해 경량형 차양막 시범 적용

4. **우천 데이터 기반 운영 정책**
   우천 주간을 정비·회수·수리에 집중하는 효율화 주간으로 운영

=================================================================
"""

print(story)

# 저장
with open('../outputs/reports/final_story.txt', 'w', encoding='utf-8') as f:
    f.write(story)

print("\n✅ 스토리 저장 완료: outputs/reports/final_story.txt")

## 5. 전체 권고사항 저장

In [ ]:
recommendations = short_term + "\n" + mid_term + "\n" + long_term

with open('../outputs/reports/recommendations.txt', 'w', encoding='utf-8') as f:
    f.write(recommendations)

print("✅ 권고사항 저장 완료: outputs/reports/recommendations.txt")

## 6. 프로젝트 완료

### 완료된 작업
- ✅ 데이터 로딩 및 탐색
- ✅ 날씨 패턴 분석
- ✅ 시공간 분석
- ✅ 집중호우 사례 분석
- ✅ 인사이트 도출 및 실행 방안 제시

### 생성된 결과물
- 📊 시각화 파일: `outputs/figures/`
- 📄 분석 리포트: `outputs/reports/`
- 💾 전처리 데이터: `data/processed/`

### 다음 단계
- 발표 자료(PPT) 작성
- 경영진 요약 리포트(PDF) 작성
- 추가 분석: 예측 모델링 (선택)